In [ ]:
!git clone https://github.com/AbdelhamidRb/tp2-sarcasme-arabe.git

Cloning into 'tp2-sarcasme-arabe'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15 (delta 3), reused 13 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), done.
Resolving deltas: 100% (3/3), done.


In [ ]:
import os, sys
os.chdir("/content/tp2-sarcasme-arabe")
sys.path.insert(0, "/content/tp2-sarcasme-arabe")
!ls

notebooks  predictions	README.md  results  src


In [ ]:
%cd /content/tp2-sarcasme-arabe
!git pull
!ls src

/content
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 1), reused 5 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 868 bytes | 868.00 KiB/s, done.
From https://github.com/AbdelhamidRb/tp2-sarcasme-arabe
   5b8242c..6d8424b  main       -> origin/main
Updating 5b8242c..6d8424b
Fast-forward
 src/data.py | 13 +++++++++++++
 src/seed.py | 11 +++++++++++
 2 files changed, 24 insertions(+)
 create mode 100644 src/data.py
 create mode 100644 src/seed.py
data.py  __init__.py  seed.py


In [ ]:
from src.seed import set_seed
from src.data import load_splits

set_seed(42)
train, val, test = load_splits()

print("=== 30 tweets de train ===")
for i, t in enumerate(train["tweet"].sample(30, random_state=0), 1):
    print(f"{i:2d}. {t!r}")

print("\n=== 30 tweets de test ===")
for i, t in enumerate(test["tweet"].sample(30, random_state=0), 1):
    print(f"{i:2d}. {t!r}")

=== 30 tweets de train ===
 1. '"#MannequinChallenge انا خلوا حياتي نفسه تثبت على مود محدد عشان انا اثبت"'
 2. 'عطر غوتشي بامبو" الجديد يحاكي الطيور https://t.co/F9PRduoecU https://t.co/eGRH39QeLq"'
 3. '"#ستبقى_حلب#ستنتصر_حلب(فدعا ربه أني مغلوب فأنتصر)والله لتنصرن ولو بعد #حين"'
 4. '"@DROS_ASEER 📚شرح أصول السُنّة للإمام أحمد✒الشيخ سلطان العرابي📍جامع الراجحي بأبها🕗عصر الجمعة ١٨-٢-١٤٣٨🕗وبعد الم… https://t.co/DqFHkhsqQ9"'
 5. 'مااحلى شعارك ومدافعك ياقنرررزز #ارسنال'
 6. '"RT @BassamJaara: الطائرات الروسية تحرق الاخضر واليابس والجنود الروس يسرحون ويمرحون  في شوارع حلب المحتلة ومدن اخرى لأنهم يشعرون بالامان!!"'
 7. 'إقالة لبيب السباعى من الطب الشرعى وتعيين د إحسان كميل جورجي بدلاً منه" الى مزبلة التاريخ وشكرا'
 8. '@aldanahk @girlposts دام انه مال هاري بوتر بستعمله 💅🏼'
 9. '"بطلبٍ من #هشام_حداد.. #رامي_عياش يحقّق حلم لين حايك https://t.co/xH63acQjcr @HichamHaddad57 @RamyAyach"'
10. '"@shihanh010 بوش والا اوباما😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂😂

In [ ]:
candidats = {
    "guillemets_echappes": r'\\"',
    "double_guillemet": r'""',
    "retour_ligne": "\n",
    "tabulation": "\t",
    "espace_insecable": "\u00a0",
    "espace_debut_ou_fin": r"^\s|\s$",
}

for nom, motif in candidats.items():
    for split_nom, df in [("train", train), ("test", test)]:
        s = df["tweet"].str.contains(motif, regex=True)
        par_label = s.groupby(df["sarcasm"]).mean().round(4).to_dict() if split_nom == "train" else {}
        print(f"{nom:22s} {split_nom:5s} freq={s.mean():.4f} {par_label}")

guillemets_echappes    train freq=0.0001 {False: 0.0001, True: 0.0}
guillemets_echappes    test  freq=0.0000 {}
double_guillemet       train freq=0.0068 {False: 0.0076, True: 0.0031}
double_guillemet       test  freq=0.0000 {}
retour_ligne           train freq=0.0001 {False: 0.0001, True: 0.0}
retour_ligne           test  freq=0.0000 {}
tabulation             train freq=0.0001 {False: 0.0001, True: 0.0}
tabulation             test  freq=0.0000 {}
espace_insecable       train freq=0.0103 {False: 0.0123, True: 0.0005}
espace_insecable       test  freq=0.0040 {}
espace_debut_ou_fin    train freq=0.0481 {False: 0.0315, True: 0.1276}
espace_debut_ou_fin    test  freq=0.3227 {}


In [ ]:
import re

patterns = {
    "url": r"https?://\S+",
    "mention": r"@\w+",
    "hashtag": r"#\w+",
    "emoji": r"[\U0001F300-\U0001FAFF\u2600-\u27BF]",
    "rires": r"(?:ه){3,}|(?:ha){2,}|(?:ha){2,}",
    "ponctuation_repetee": r"([!?.])\1+",
    "tatweel": "\u0640",
    "chiffres_indiens": r"[\u0660-\u0669]",
    "latin": r"[a-zA-Z]",
    "allongement_lettre": r"(.)\1{2,}",
}

for nom, p in patterns.items():
    s_tr = train["tweet"].str.contains(p, regex=True)
    s_te = test["tweet"].str.contains(p, regex=True)
    print(f"{nom:20s} train={s_tr.mean():.3f} test={s_te.mean():.3f} | "
          f"sarc=1: {s_tr[train.sarcasm].mean():.3f}  sarc=0: {s_tr[~train.sarcasm].mean():.3f}")

url                  train=0.381 test=0.147 | sarc=1: 0.145  sarc=0: 0.430
mention              train=0.234 test=0.008 | sarc=1: 0.225  sarc=0: 0.235
hashtag              train=0.472 test=0.329 | sarc=1: 0.405  sarc=0: 0.486
emoji                train=0.120 test=0.163 | sarc=1: 0.115  sarc=0: 0.121
rires                train=0.012 test=0.033 | sarc=1: 0.042  sarc=0: 0.006
ponctuation_repetee  train=0.162 test=0.242 | sarc=1: 0.186  sarc=0: 0.158
tatweel              train=0.046 test=0.073 | sarc=1: 0.043  sarc=0: 0.046
chiffres_indiens     train=0.024 test=0.026 | sarc=1: 0.017  sarc=0: 0.026
latin                train=0.554 test=0.239 | sarc=1: 0.349  sarc=0: 0.597
allongement_lettre   train=0.164 test=0.340 | sarc=1: 0.232  sarc=0: 0.150


/tmp/ipykernel_2142/2152398039.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  s_tr = train["tweet"].str.contains(p, regex=True)
/tmp/ipykernel_2142/2152398039.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  s_te = test["tweet"].str.contains(p, regex=True)


In [ ]:
def normaliser_simple(s: str) -> str:
    s = re.sub(r"https?://\S+|@\w+", "", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

train["norm"] = train["tweet"].map(normaliser_simple)
test["norm"] = test["tweet"].map(normaliser_simple)

print("doublons exacts dans train :", train["tweet"].duplicated().sum())
print("doublons normalisés dans train :", train["norm"].duplicated().sum())

chevauche = train["norm"].isin(set(test["norm"]))
print("tweets de train présents dans test :", chevauche.sum())

# Doublons aux labels contradictoires dans train
g = train.groupby("norm")["sarcasm"].nunique()
print("textes normalisés aux labels contradictoires :", (g > 1).sum())

doublons exacts dans train : 54
doublons normalisés dans train : 59
tweets de train présents dans test : 4
textes normalisés aux labels contradictoires : 13


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11293 entries, 0 to 11292
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tweet      11293 non-null  object
 1   sarcasm    11293 non-null  bool  
 2   sentiment  11293 non-null  object
 3   dialect    11293 non-null  object
 4   id         11293 non-null  int64 
 5   norm       11293 non-null  object
dtypes: bool(1), int64(1), object(4)
memory usage: 452.3+ KB
